# C1: Classification and Regression Trees (CART)

## Introduction to CART

Classification and Regression Trees (CART) is a predictive modeling algorithm used in machine learning. It explains how a target variable's values can be predicted based on other variables. It is a decision tree learning technique that produces either classification or regression trees, depending on whether the dependent variable is categorical or numeric, respectively.

### Key Concepts
- **Root Node**: The starting point of the tree. It represents the entire dataset, which is then split into two or more homogeneous sets.
- **Internal Node**: A node where the data is split based on a feature and a condition.
- **Leaf Node (Terminal Node)**: The final node that represents the outcome (class label or numerical value). It does not split further.
- **Splitting**: The process of dividing a node into two or more sub-nodes.
- **Pruning**: The process of removing sub-nodes of a decision node to prevent overfitting.

![Decision Tree Terminology](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2F365datascience.com%2Fresources%2Fblog%2Frr6cuudl59r-decision-trees-image1.png&f=1&nofb=1&ipt=0847520715bcddbfd0315da47ec77c711e4209f0e0351fb9286d1cfdc3e679a0)
*(Source: Wikimedia Commons - Decision Tree)*

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import make_classification, make_regression

# Set random seed for reproducibility
np.random.seed(42)
print("Libraries imported successfully.")

## Part 1: Classification Trees

Classification trees are used when the target variable is categorical (e.g., Yes/No, Spam/Not Spam).

### Impurity Measures: Gini vs Entropy

To decide where to split the data, the algorithm uses impurity measures. The goal is to maximize the purity of the child nodes.

- **Gini Impurity**: Measures the probability of misclassifying a randomly chosen element. It ranges from 0 (pure) to 0.5 (impure) for binary classification.
  $$ Gini = 1 - \sum (p_i)^2 $$

- **Entropy**: Measures the level of disorder or uncertainty. It ranges from 0 (pure) to 1 (impure).
  $$ Entropy = - \sum p_i \log_2(p_i) $$

#### Diagram: Gini vs Entropy
![Gini vs Entropy](https://media.geeksforgeeks.org/wp-content/uploads/20251031102654718442/entropy_vs_gini_impurity.webp)

*(Source: GeeksforGeeks - Comparing Gini & Entropy)*

As you can see, both metrics behave similarly. Gini is often computationally faster because it doesn't calculate logs.


### Classification Example: Breast Cancer Dataset

Let's train a Decision Tree Classifier on the Wisconsin Breast Cancer dataset.

In [ ]:
# Load dataset
df_cancer = pd.read_csv('../../00-data/wisconsin_breat_cancer.csv')

# Display first few rows
df_cancer.head()

In [ ]:
# Prepare X and y
# Assuming 'diagnosis' is the target. Map M to 1 and B to 0 if necessary, or let sklearn handle it if it supports strings (better to map)
if 'diagnosis' in df_cancer.columns:
    df_cancer['diagnosis'] = df_cancer['diagnosis'].map({'M': 1, 'B': 0})
    X_cls = df_cancer.drop(['id', 'diagnosis', 'Unnamed: 32'], axis=1, errors='ignore') # Drop ID and empty col if exists
    y_cls = df_cancer['diagnosis']
else:
    # Fallback if column names differ, likely it is the first or second column
    X_cls = df_cancer.iloc[:, 2:]
    y_cls = df_cancer.iloc[:, 1].map({'M': 1, 'B': 0})

# Split data
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Train Decision Tree Classifier (using Gini by default)
clf = DecisionTreeClassifier(criterion='gini', random_state=42)
clf.fit(X_train_cls, y_train_cls)

In [ ]:
# Visualize the Tree
plt.figure(figsize=(20,10))
plot_tree(clf, feature_names=X_cls.columns, class_names=['Benign', 'Malignant'], filled=True, max_depth=3)
plt.show()

### Classification Metrics

To evaluate our classification model, we use several metrics:

1.  **Confusion Matrix**: Shows True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).
    ![Confusion Matrix](https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/Precisionrecall.svg/1024px-Precisionrecall.svg.png)

2.  **ROC Curve (Receiver Operating Characteristic)**: Plots True Positive Rate vs False Positive Rate at different thresholds.
3.  **AUC (Area Under the Curve)**: Represents the degree of separability. Higher AUC (close to 1) means better performance.

![ROC AUC Curve](https://upload.wikimedia.org/wikipedia/commons/thumb/1/13/Roc_curve.svg/1024px-Roc_curve.svg.png)
*(Source: Wikimedia Commons - ROC Curve)*

In [ ]:
# Make predictions
y_pred_cls = clf.predict(X_test_cls)

# Confusion Matrix
cm = confusion_matrix(y_test_cls, y_pred_cls)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Classification Report
print(classification_report(y_test_cls, y_pred_cls))

In [ ]:
# Plot ROC Curve
from sklearn.metrics import roc_curve, auc

y_prob_cls = clf.predict_proba(X_test_cls)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test_cls, y_prob_cls)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

### Exercise: Classification
1.  Train a new Decision Tree using `criterion='entropy'`.
2.  Compare its accuracy with the Gini model.
3.  Which metric performed better, or are they similar?

---

## Part 2: Regression Trees

Regression trees are used when the target variable is continuous (e.g., House Price, Temperature, MPG).

### Splitting Criterion: MSE
For regression, CART splits the node to minimize the **Variance** or **Mean Squared Error (MSE)** of the target values in the child nodes.

### Regression Metrics
To evaluate regression models, we use:

- **Mean Absolute Error (MAE)**: Average of absolute errors. Robust to outliers.
- **Mean Squared Error (MSE)**: Average of squared errors. Penalizes large errors heavily.
- **Root Mean Squared Error (RMSE)**: Square root of MSE. Easier to interpret (same units as target).
- **R-squared ($R^2$)**: Proportion of variance explained by the model. 1 is perfect, 0 is baseline.

![Regression Metrics Diagram](https://upload.wikimedia.org/wikipedia/commons/thumb/e/ed/Residuals_for_Linear_Regression_Fit.png/1024px-Residuals_for_Linear_Regression_Fit.png)
*(Source: Wikimedia Commons - Residuals)*

### Regression Example: Auto MPG Dataset

We will predict the `mpg` (miles per gallon) of cars.

In [ ]:
# Load dataset
df_mpg = pd.read_csv('../../00-data/auto_mpg.csv')

# Basic preprocessing: Drop rows with missing values ('?' often used in this dataset)
df_mpg = df_mpg.replace('?', np.nan).dropna()

# Prepare X and y
X_reg = pd.get_dummies(df_mpg.drop(columns=['mpg', 'car name'], errors='ignore'), drop_first=True, dtype=float)   # this copy's origin is text
y_reg = df_mpg['mpg']

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Train Decision Tree Regressor
reg = DecisionTreeRegressor(criterion='squared_error', random_state=42) # squared_error is MSE
reg.fit(X_train_reg, y_train_reg)

In [ ]:
# Visualize Regression Tree (Top 3 levels)
plt.figure(figsize=(20,10))
plot_tree(reg, feature_names=X_reg.columns, filled=True, max_depth=3)
plt.show()

In [ ]:
# Evaluate
y_pred_reg = reg.predict(X_test_reg)

mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.2f}")

### Exercise: Regression
1.  Create a new `DecisionTreeRegressor`.
2.  Try setting `max_depth=5` and compare the $R^2$ score with the unbounded tree.
3.  Does limiting the depth improve or worsen performance on the test set?

---

## Part 3: Overfitting, Underfitting, and Bias-Variance Tradeoff

Decision trees are prone to **overfitting**. This means they memorize the training data, including the noise, creating a very complex tree.

### Definitions
- **Overfitting (High Variance)**: The model learns the training data too well but fails to generalize to new data. The tree is too deep.
- **Underfitting (High Bias)**: The model is too simple to capture the underlying patterns. The tree is too shallow.
- **Balance (Sweet Spot)**: Minimal total error (Bias + Variance).

### Diagrams

#### Overfitting vs Underfitting (Data Fit)
![Overfitting vs Underfitting](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmedia.geeksforgeeks.org%2Fwp-content%2Fcdn-uploads%2F20190523171258%2Foverfitting_2.png&f=1&nofb=1&ipt=b1b3946ce1d4f9dfdfeb725904aa8c9ac474f4f4e59acb80a2a3685cc9386fa9)
*(Source: Wikimedia Commons - Overfitting)*

#### Bias-Variance Tradeoff (Total Error)
![Bias Variance Tradeoff](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmlvidhya.com%2Fwp-content%2Fuploads%2F2023%2F12%2FBias-Variance-Tradeoff-768x768.jpg&f=1&nofb=1&ipt=146664aee60618849aa5199224acbcc7011d09570b530ace9179f37d4e740cc5)

*(Source: Wikimedia Commons - Bias Variance Tradeoff)*

### Demonstration
Let's visualize how changing `max_depth` affects accuracy (bias vs variance).

In [ ]:
depths = range(1, 21)
train_scores = []
test_scores = []

for d in depths:
    # Train model with max_depth=d
    clf_depth = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf_depth.fit(X_train_cls, y_train_cls)
    
    # Record scores
    train_scores.append(clf_depth.score(X_train_cls, y_train_cls))
    test_scores.append(clf_depth.score(X_test_cls, y_test_cls))

# Plot
plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, label='Training Accuracy')
plt.plot(depths, test_scores, label='Testing Accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Overfitting Demonstration: Accuracy vs Tree Depth')
plt.legend()
plt.xticks(depths)
plt.grid(True)
plt.show()

**Analysis:**
- **Low Depth (Underfitting)**: Both Training and Testing accuracy are low.
- **High Depth (Overfitting)**: Training accuracy reaches nearly 100%, but Testing accuracy stays flat or drops. Using a depth of 20 is likely overfitting.
- **Sweet Spot**: Where Testing accuracy is highest before it diverges from the Training accuracy.

---

## Part 4: Practice with Simulated Data

Now it's your turn to build models from scratch using synthetic data.

### Exercise A: Simulated Classification

1.  **Generate Data**: Use the code below to generate a classification dataset.
2.  **Split**: Split into training and testing sets.
3.  **Model**: Train a `DecisionTreeClassifier` with `max_depth=4`.
4.  **Evaluate**: Plot the confusion matrix and print the accuracy.

In [ ]:
# 1. Generate Data
X_sim_cls, y_sim_cls = make_classification(n_samples=1000, n_features=10, n_informative=5, n_redundant=2, random_state=42)

# TODO: Write your code here for splitting, training, and evaluation

### Exercise B: Simulated Regression

1.  **Generate Data**: Use the code below to generate a regression dataset.
2.  **Split**: Split into training and testing sets.
3.  **Model**: Train a `DecisionTreeRegressor`.
4.  **Evaluate**: Calculate RMSE and R2 score.

In [ ]:
# 1. Generate Data
X_sim_reg, y_sim_reg = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)

# TODO: Write your code here for splitting, training, and evaluation